# Setup
Select a GPU runtime and run all cells. When prompted, authorize the Google account you used to open Colab. The notebook uses `BDD100K_Project/bdd100k_subset.zip` if present, or `images.zip` and `labels.zip` in its `bdd100k_data` folder. If the project folder is under **Shared with me**, add a shortcut to it at the top level of My Drive.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/Cs117-Benchmark")
if REPO_ROOT.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_ROOT, check=True)
else:
    subprocess.run([
        "git", "clone", "https://github.com/Landon94/Cs117-Benchmark.git",
        str(REPO_ROOT),
    ], check=True)

os.chdir(REPO_ROOT)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", "requirements-checkin.txt",
], check=True)


In [ ]:
from tempfile import TemporaryDirectory
import zipfile

from google.colab import drive

# If authorization fails, rerun this cell and complete the Google prompt.
drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/BDD100K_Project")
subset_zip = project / "bdd100k_subset.zip"

if subset_zip.is_file():
    data_root = Path("/content/bdd100k_subset")
    if not data_root.exists():
        with TemporaryDirectory(dir=data_root.parent) as temp_dir:
            extracted = Path(temp_dir) / "dataset"
            with zipfile.ZipFile(subset_zip) as archive:
                archive.extractall(extracted)
            extracted.rename(data_root)
    dataset_args = ["--subset", str(data_root), "--split", "test"]
    print(f"Using prepared subset at {data_root}")
else:
    images_zip = project / "bdd100k_data" / "images.zip"
    labels_zip = project / "bdd100k_data" / "labels.zip"
    missing = [str(path) for path in (images_zip, labels_zip) if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Prepared subset missing at {subset_zip}. "
            f"Also missing Drive archives at {', '.join(missing)}. "
            "Check the BDD100K_Project folder or shortcut in My Drive."
        )
    # Our held-out test manifest lists images from the official validation split.
    dataset_args = [
        "--images", str(images_zip), "--labels", str(labels_zip),
        "--split", "val", "--manifest", "splits/test.txt",
    ]
    print(f"Using images.zip and labels.zip from {images_zip.parent}")


# Baseline runs
Runs each pretrained model on the held-out test split (1,000 images). Use a GPU runtime (Runtime > Change runtime type > T4).

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")

def run_baseline(model, result_name):
    result = subprocess.run([
        sys.executable, "-m", "src.run_checkin", *dataset_args,
        "--model", model,
        "--output", f"results/{result_name}_test.json",
        "--predictions-output", f"results/{result_name}_test_predictions.json",
    ], cwd=REPO_ROOT, stderr=subprocess.PIPE, text=True)
    if result.returncode:
        details = result.stderr.strip() or "The runner produced no error output."
        raise RuntimeError(f"{model} exited with status {result.returncode}\n{details}")
    if result.stderr:
        print(result.stderr, end="")


In [ ]:
# YOLO11m, COCO-pretrained, no fine-tuning
run_baseline("yolo11m.pt", "yolo11m")


In [ ]:
# RT-DETR-L, COCO-pretrained, no fine-tuning
run_baseline("rtdetr-l.pt", "rtdetr_l")


In [ ]:
import json
for name in ("yolo11m", "rtdetr_l"):
    r = json.load(open(f"results/{name}_test.json"))
    print(name, "mAP@0.5:0.95 =", round(r["measurements"]["map50_95"], 4),
          "| mean ms/img =", r["timing"]["mean_inference_ms"])
